In [ ]:
# 两层网络实现波士顿房价预测模型
import numpy as np 
import pandas as pd

def loaddata():
	boston = "http://lib.stat.cmu.edu/datasets/boston"
	raw_df = pd.read_csv(boston, sep=r"\s+", skiprows=22, header=None)
	data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :3]])
	# target = raw_df.values[1::2, 2]
	feature_names = [
		'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE',
		'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'
	]
	feature_num = len(feature_names)
	print("feature_num = ",feature_num)

	ratio = 0.8
	offset = int(data.shape[0]*ratio)
	training  = data[:offset]
	maximums,minimums = training.max(axis=0),training.min(axis=0)
	for i in range(feature_num):
		data[:,i] = (data[:,i] - minimums[i])/(maximums[i]-minimums[i])

	training = data[:offset]
	test = data[offset:]
	return training,test

class towlayernet():
	def __init__(self,num_of_weights,hidden_dim=13):# 隐藏层可以是任意维度的 hiden layer could be any dimensions
		np.random.seed(0)
		self.w1 = np.random.randn(num_of_weights, 1) #randn(d1,d2,...dn) 生成d1...dn的具有正态分布的数组，如randn(3,3)随机生成3*3的矩阵
		self.w2 = np.random.randn(num_of_weights, 1)
		self.b1 = np.zeros((1, hidden_dim))
		self.b2 = 0

	def forward(self, x):
		self.z1 = np.dot(x, self.w1) + self.b1
		self.a1 = self.z1
		self.z2 = np.dot(self.a1,self.w2) + self.b2
		return self.z2

	def loss(self,y_hat,y):
		error = y_hat - y
		return np.mean(error**2)
	
	def backword(self,y_hat,y):
		N = y.shape[0]

		# 输出层
		delta2 = (y_hat-y)/N #(N,1)
		grad_w2 = np.dot(self.z1.T,delta2)
		grad_b2 = np.sum(delta2)

		# 结果传入隐藏层
		delta1 = np.dot(delta2,self.w2.T) # (N,13)
		grad_w1 = np.dot(self.x.T,delta1)
		grad_b1 = np.sum(delta1,aixs=0,keepdims=True)

		return grad_w1,grad_b1,grad_w2,grad_b2
	
	def updata(self,grads,eta=0.01):
		grad_w1,grad_b1,grad_w2,grad_b2 = grads
		self.w1 -= eta * grad_w1
		self.b1 -= eta * grad_b1
		self.w2 -= eta * grad_w2
		self.b2 -= eta * grad_b2

	def train(self,x,y,iteration=1000,eta=0.01):
		losses = []
		for i in range(iteration):
			y_hat = self.forward(x)
			loss = self.loss(y_hat,y)

			grads = self.backword(y_hat,y)
			self.updata(grads,eta=eta)
			losses.append(loss)
			if i % 100 == 0:
				print(f"iter {i}, loss {loss:.4f}")
		
		return losses

net = towlayernet(13)
train_data, test_data = loaddata()
x = train_data[:,:-1]
y = train_data[:,-1:]
iterations = 1000 

losses = net.train(x,y,iteration=iterations)



